# Further fine-tune the fire/smoke model on the LARGE dataset

Fine-tunes [`models/fire/best.pt`](../models/fire/best.pt) (HF **YOLO26-S**, classes `fire(0)/other(1)/smoke(2)`) on the big local Roboflow export `ready_fire_smoke_dataset.yolov8` (12,799 train imgs) + the curated `default-other` background negatives.

- Plan: [`plans/fire-model-large-finetune-colab.md`](../plans/fire-model-large-finetune-colab.md)
- Dataset prep (run once, locally): `python dev_scripts/prep_fire_large_dataset.py --ready dataset/ready_fire_smoke_dataset.yolov8 --negatives dataset/default-other --out dataset/large_finetune --val-frac 0.05 --seed 0 --zip`
  → produces `dataset/large_finetune_colab.zip` (train 12,170 / val 629, all 640×640).
- **Class contract:** fire=0 / other=1 / smoke=2. The export is already index-aligned, so labels are **not** remapped; only the `data.yaml` names are set. `other` is trained but ignored in production.


In [ ]:
# Cell 1 - runtime + install a current ultralytics (YOLO26 needs >= ~8.3)
!nvidia-smi
!pip install -q --upgrade ultralytics

import ultralytics, torch
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


## Configuration

Set the `DRIVE_*` paths if you keep the zip / `best.pt` on Google Drive (recommended for the ~600 MB dataset zip). Set either to `""` and the matching step will pop up a file upload dialog instead.

In [ ]:
# Cell 2 - EDIT THIS

DRIVE_ZIP = '/content/drive/MyDrive/mazr3a/large_finetune_colab.zip'  # dataset zip (or '')
DRIVE_BEST = '/content/drive/MyDrive/mazr3a/models_fire_best.pt'       # base checkpoint (or '')

# --- training hyperparameters (tune freely) ---
EPOCHS   = 50     # large set -> more epochs than the Abonia smoke run; patience will stop early
BATCH    = 16     # reduce to 8 if OOM on a T4; raise on A100/L4
IMGSZ    = 640    # matches production input
FREEZE   = 10     # keep early backbone (faster, less forgetting); 0 = full fine-tune
LR0      = 0.001  # low LR for a pretrained fine-tune
PATIENCE = 15     # early stop on the held-out val split
PROJECT  = 'fire_large_finetune'
NAME     = 'run1'
WORKDIR  = '/content/fire_large'


In [ ]:
# Cell 3 - (only needed if you use Drive paths) mount Drive
if DRIVE_ZIP or DRIVE_BEST:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive not required - using file upload dialogs.')


In [ ]:
# Cell 4 - get + extract the dataset, then fix data.yaml path for Colab
import glob, os, shutil, zipfile

os.makedirs(WORKDIR, exist_ok=True)

data_zip = None
if DRIVE_ZIP and os.path.exists(DRIVE_ZIP):
    data_zip = DRIVE_ZIP
else:
    if DRIVE_ZIP:
        print('Drive zip not found - uploading instead.')
    from google.colab import files
    uploaded = files.upload()
    data_zip = next(iter(uploaded))
print('data zip:', data_zip)

with zipfile.ZipFile(data_zip) as z:
    z.extractall(WORKDIR)

# the zip top-level contains train/ val/ data.yaml prep_report.txt
yaml_path = os.path.join(WORKDIR, 'data.yaml')
assert os.path.isfile(yaml_path), 'data.yaml not found after extract: ' + yaml_path

# rewrite the absolute local `path:` to the Colab location
lines = []
for ln in open(yaml_path, encoding='utf-8'):
    if ln.startswith('path:'):
        lines.append(f'path: {WORKDIR}\n')
    else:
        lines.append(ln)
open(yaml_path, 'w', encoding='utf-8').writelines(lines)

print('data.yaml ->', yaml_path)
print(open(yaml_path, encoding='utf-8').read())


In [ ]:
# Cell 5 - verify the extracted layout + class coverage
import collections, os

def tally(img_dir, lbl_dir):
    n = n_box = 0
    cls = collections.Counter()
    for f in sorted(os.listdir(img_dir)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
            continue
        n += 1
        lb = os.path.join(lbl_dir, os.path.splitext(f)[0] + '.txt')
        if os.path.isfile(lb):
            for row in open(lb, encoding='utf-8'):
                if row.split():
                    cls[int(float(row.split()[0]))] += 1
                    n_box += 1
    return n, n_box, cls

for split in ('train', 'val'):
    n, nb, cls = tally(os.path.join(WORKDIR, split, 'images'), os.path.join(WORKDIR, split, 'labels'))
    names = {0: 'fire', 1: 'other', 2: 'smoke'}
    print(f'{split}: images={n} boxes={nb} per-class=' +
          ', '.join(f'{names[k]}={v}' for k, v in sorted(cls.items())))


In [ ]:
# Cell 6 - get the BASE checkpoint (models/fire/best.pt) and verify class order
base_pt = None
if DRIVE_BEST and os.path.exists(DRIVE_BEST):
    base_pt = DRIVE_BEST
else:
    if DRIVE_BEST:
        print('Drive best.pt not found - uploading instead.')
    from google.colab import files
    up = files.upload()
    base_pt = next(k for k in up if k.endswith('.pt'))
print('base:', base_pt)

from ultralytics import YOLO
m = YOLO(base_pt)
assert list(m.names.values()) == ['fire', 'other', 'smoke'], m.names
print('class order OK ->', m.names)


In [ ]:
# Cell 7 - FINE-TUNE on the large dataset (GPU)
m.train(data=os.path.join(WORKDIR, 'data.yaml'),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
        project=PROJECT, name=NAME,
        freeze=FREEZE, lr0=LR0, patience=PATIENCE,
        plots=True, exist_ok=True, verbose=True)


In [ ]:
# Cell 8 - post-train: verify .names, keep a clean candidate, zip + download
import os, shutil, zipfile

best = os.path.join(PROJECT, NAME, 'weights', 'best.pt')
last = os.path.join(PROJECT, NAME, 'weights', 'last.pt')
print('best exists:', os.path.exists(best), '->', best)

fin = YOLO(best if os.path.exists(best) else last)
print('final .names:', fin.names)
assert list(fin.names.values()) == ['fire', 'other', 'smoke']

cand = 'best_finetuned_large.pt'
shutil.copy(best if os.path.exists(best) else last, cand)

# optional: bundle results for download
zname = 'fire_large_finetune_results.zip'
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(cand, cand)
    for root, _dirs, files in os.walk(os.path.join(PROJECT, NAME)):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, PROJECT))
print('results zip ->', zname)

from google.colab import files
files.download(zname)
files.download(cand)


## Next steps after the run

1. Save `best_finetuned_large.pt` back into this repo as `dataset/large_finetune/best_finetuned_large.pt` (git-ignored).
2. Run the local eval + class-order check in [`plans/fire-model-large-finetune-colab.md`](../plans/fire-model-large-finetune-colab.md) §5.
3. If metrics justify it, archive as a versioned candidate (`models/fire/versions/`, per [`plans/model-versioning.md`](../plans/model-versioning.md)) and only then consider promotion + OpenVINO export + on-host verification (`ssh.mazr3a.garden`).